In [5]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install --upgrade pip

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***") # 키 전체가 출력되지 않도록 앞 4자만 확인

7FD6***


In [14]:
import requests
import json
def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    """우리말샘 개방형 사전 API를 통해 단어를 검색하고 결과를 dict로 반환합니다."""
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word"
    }
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()



우리말샘 API 엔드포인트에 맞춰 필요한 쿼리 매개변수와 인증키를 requests.get 메서드로 전달하도록 함수 구현, 네트워크 지연으로 인한 무한 대기를 방지하기 위해 timeout=10, raise_for_status()를 통해 HTTP 요청이 성공했는지 검증한 후 안전하게 JSON 데이터를 파싱하여 반환하도록 처리

In [16]:
data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


ensure_ascii=False 옵션을 제외하면, 한글 텍스트가 사람이 읽을 수 있는 글자 대신에 유니코드 이스케이프 시퀀스로 변환되어 출력됨

In [17]:
total: int = int(data["channel"]["total"])
n: int = len(data["channel"]["item"])
print(f"총 {total}건, 이 페이지 {n}건")

for item in data["channel"]["item"][:5]:
    word: str = item["word"]
    
    senses: list = item.get("sense", [])
    first_sense: dict = senses[0] if senses else {}
    
    pos: str = item.get("pos") or first_sense.get("pos") or "품사 없음"
    
    definition: str = first_sense.get("definition", "뜻풀이 없음")
    definition_truncated: str = definition[:40]
    
    print(f"{word} ({pos}) -> {definition_truncated}")

총 328건, 이 페이지 10건
김치 (명사) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


사전 데이터 중 상위 5개 항목을 순회하면서 단어명, 품사, 뜻풀이 데이터를 추출. 특히 특정 항목에 품사 정보가 누락되어 발생할 수 있는 오류를 방지하기 위해 dict.get 메서드를 사용하여 안정성을 높임

In [20]:
import time
words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥"
]

all_items: list[dict] = []

for q in words:
    result_data = search_word(q)
    
    total_count: int = int(result_data["channel"]["total"])
    print(f"{q}: {total_count}건")
    
    items: list[dict] = result_data["channel"]["item"]
    all_items.extend(items)
    
    time.sleep(0.3)

print("\n" + "="*30 + "\n")

pos_list: list[str] = [item.get("pos") or "(미상)" for item in all_items]

pos_counter = Counter(pos_list)
top_3_pos = pos_counter.most_common(3)

print("품사 빈도 상위 3개:")
for pos, count in top_3_pos:
    print(f"{pos}: {count}회")

ConnectTimeout: HTTPSConnectionPool(host='opendict.korean.go.kr', port=443): Max retries exceeded with url: /api/search?key=7FD6C67BBAD045935F855488563D6560&q=%EA%B9%80%EC%B9%98&req_type=json&num=10&start=1&type1=word (Caused by ConnectTimeoutError(<HTTPSConnection(host='opendict.korean.go.kr', port=443) at 0x755e5f387440>, 'Connection to opendict.korean.go.kr timed out. (connect timeout=10)'))

주어진 8개의 음식 관련 단어 리스트를 순회하며 API를 연속 요청하되, 서버 과부하를 막기 위해 요청 사이에 time.sleep(0.3)을 설정하고, 이후 수집된 모든 단어 항목들의 품사 필드를 취합하고 collections.Counter를 활용하여 가장 많이 등장한 품사 상위 3개를 산출함
    # 관찰: 가장 흔하게 나타난 품사는 명사로, 이는 검색어로 사용한 단어들이 모두 '음식'이라는 구체적인 대상을 가리키는 사물의 명칭, 즉 명사이기 때문이다. 이들과 결합하여 파생된 주변 어휘들(예: 김칫국, 국수틀 등) 역시 사전 내에서 대부분 명사로 등록되어 있기 때문에 명사가 가장 많이 나타난다.